# Gold Layer - Player Consolidation

This notebook creates the **gold layer** for FantasAI by consolidating player data from multiple sources into unified, analytics-ready tables.

## Objectives

1. **Unified Player Dimension** - Create a master player registry with a single player_id across all sources
2. **Player ID Mapping** - Map source-specific IDs (ESPN, nflverse, Sleeper, API-Sports.io) to master IDs
3. **Data Quality** - Resolve conflicts, handle duplicates, and ensure consistency
4. **Gold Layer Tables** - Clean, consolidated weekly stats ready for analytics and ML

## Tables Created

| Table | Type | Purpose |
|-------|------|----------|
| `gold_player_dim` | Dimension | Master player registry with unified IDs |
| `gold_player_id_mapping` | Mapping | Cross-reference of source IDs to master IDs |
| `gold_weekly_stats` | Fact | Consolidated weekly stats with master player IDs |

## Data Sources

* **Bronze/Silver:** `main.fantasai.silver_weekly_stats`
* **Sources:** `espn_public`, `nflverse`, `sleeper`, `api_sports`, `fantasai`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import json
from datetime import datetime

print("✓ Imports loaded")
print(f"✓ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Configuration
CATALOG = "main"
SCHEMA = "fantasai"

print(f"\n✓ Catalog: {CATALOG}")
print(f"✓ Schema: {SCHEMA}")

In [0]:
# Analyze the current state of player data across all sources
print("="*70)
print("PLAYER DATA ANALYSIS ACROSS SOURCES")
print("="*70)

# Load silver data
silver_df = spark.table(f"{CATALOG}.{SCHEMA}.silver_weekly_stats")

print("\n1. Records by Source:")
source_counts = silver_df.groupBy("source").agg(
    F.count("*").alias("total_records"),
    F.countDistinct("player_id").alias("unique_players"),
    F.countDistinct("season").alias("seasons"),
    F.countDistinct("week").alias("weeks")
).orderBy(F.desc("total_records"))

display(source_counts)

print("\n2. Sample Player IDs by Source:")
# Extract player names from JSON stats to understand naming conventions
for source in ['espn_public', 'nflverse', 'sleeper', 'api_sports', 'fantasai']:
    source_sample = silver_df.filter(F.col("source") == source).limit(3)
    
    if source_sample.count() > 0:
        print(f"\n{source.upper()}:")
        for row in source_sample.collect():
            try:
                stats = json.loads(row.stats)
                player_name = stats.get('player_name', 'Unknown')
                position = stats.get('position', 'Unknown')
                team = stats.get('team', 'Unknown')
                print(f"  ID: {row.player_id[:20]}... | Name: {player_name} | Pos: {position} | Team: {team}")
            except:
                print(f"  ID: {row.player_id[:20]}... | [Could not parse stats]")

print("\n3. Player Overlap Analysis:")
# Count how many unique player_id values exist across all sources
total_unique = silver_df.select("player_id", "source").distinct().count()
print(f"Total unique (player_id, source) combinations: {total_unique}")

# Count players who appear in multiple sources (by name matching)
print("\nAnalyzing cross-source player matches...")

In [0]:
# Extract player metadata from JSON stats field for matching
print("="*70)
print("EXTRACTING PLAYER METADATA FROM STATS")
print("="*70)

def extract_player_info(stats_json):
    """Extract player name, position, team from JSON stats."""
    try:
        stats = json.loads(stats_json)
        
        # Handle different JSON structures per source
        player_name = (
            stats.get('player_name') or 
            stats.get('full_name') or
            stats.get('name') or
            'Unknown'
        )
        
        position = (
            stats.get('position') or
            stats.get('pos') or
            'UNK'
        )
        
        team = (
            stats.get('team') or
            stats.get('team_abbr') or
            'UNK'
        )
        
        return (player_name, position, team)
    except:
        return ('Unknown', 'UNK', 'UNK')

extract_udf = F.udf(extract_player_info, StructType([
    StructField("player_name", StringType()),
    StructField("position", StringType()),
    StructField("team", StringType())
]))

# Create enriched view with extracted metadata
silver_enriched = silver_df.withColumn(
    "parsed",
    extract_udf(F.col("stats"))
).select(
    "player_id",
    "source",
    "season",
    "week",
    "fantasy_points",
    "stats",
    "ingested_at",
    F.col("parsed.player_name").alias("player_name"),
    F.col("parsed.position").alias("position"),
    F.col("parsed.team").alias("team")
)

silver_enriched.createOrReplaceTempView("silver_enriched")

print("\n✓ Created silver_enriched temp view with extracted metadata")
print("\nSample records:")
display(silver_enriched.limit(10))

In [0]:
# Create unified player dimension table
print("="*70)
print("CREATING GOLD PLAYER DIMENSION TABLE")
print("="*70)

# Strategy: Use player name + position as the matching key
# Generate a master player_id (UUID) for each unique player

# Normalize player names for better matching
def normalize_name(name):
    """Normalize player names for matching."""
    if not name or name == 'Unknown':
        return 'UNKNOWN'
    # Remove suffixes, lowercase, remove extra spaces
    name = name.upper().strip()
    name = name.replace(' JR.', '').replace(' SR.', '').replace(' III', '').replace(' II', '')
    name = name.replace('.', '').replace(',', '')
    return name

normalize_name_udf = F.udf(normalize_name, StringType())

# Get unique players across all sources
players_raw = spark.sql("""
    SELECT DISTINCT
        player_name,
        position,
        team,
        source
    FROM silver_enriched
    WHERE player_name != 'Unknown'
        AND player_name IS NOT NULL
""")

# Normalize and deduplicate
players_normalized = players_raw.withColumn(
    "player_name_normalized",
    normalize_name_udf(F.col("player_name"))
)

# Create master player records
# Group by normalized name + position to identify unique players
players_master = players_normalized.groupBy(
    "player_name_normalized",
    "position"
).agg(
    # Take the most common name spelling
    F.first("player_name").alias("display_name"),
    # Take the most recent team
    F.first("team").alias("current_team"),
    # List all sources where this player appears
    F.collect_set("source").alias("sources"),
    F.count("*").alias("source_count")
).withColumn(
    # Generate master player ID using hash of normalized name + position
    "master_player_id",
    F.sha2(F.concat(F.col("player_name_normalized"), F.lit("_"), F.col("position")), 256)
).withColumn(
    "created_at",
    F.current_timestamp()
).select(
    "master_player_id",
    "display_name",
    "player_name_normalized",
    "position",
    "current_team",
    "sources",
    "source_count",
    "created_at"
)

print(f"\n✓ Identified {players_master.count()} unique players")

# Write to gold player dimension table
players_master.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.gold_player_dim"
)

print(f"\n✓ Created table: {CATALOG}.{SCHEMA}.gold_player_dim")

# Show sample
print("\nSample players:")
display(spark.table(f"{CATALOG}.{SCHEMA}.gold_player_dim").orderBy(F.desc("source_count")).limit(20))

In [0]:
# Create mapping table from source-specific IDs to master IDs
print("="*70)
print("CREATING PLAYER ID MAPPING TABLE")
print("="*70)

# Load player dimension
player_dim = spark.table(f"{CATALOG}.{SCHEMA}.gold_player_dim")

# Load silver enriched data
silver_enriched = spark.sql("""
    SELECT DISTINCT
        player_id as source_player_id,
        source,
        player_name,
        position,
        team
    FROM silver_enriched
    WHERE player_name != 'Unknown'
        AND player_name IS NOT NULL
""")

# Normalize names in silver data
silver_with_normalized = silver_enriched.withColumn(
    "player_name_normalized",
    normalize_name_udf(F.col("player_name"))
)

# Join to player dimension on normalized name + position
id_mapping = silver_with_normalized.join(
    player_dim,
    (silver_with_normalized.player_name_normalized == player_dim.player_name_normalized) &
    (silver_with_normalized.position == player_dim.position),
    "left"
).select(
    F.col("master_player_id"),
    F.col("source"),
    F.col("source_player_id"),
    silver_with_normalized.player_name.alias("source_player_name"),
    silver_with_normalized.position.alias("position"),
    silver_with_normalized.team.alias("team"),
    F.current_timestamp().alias("created_at")
).distinct()

# Check for unmapped players (should be minimal)
unmapped_count = id_mapping.filter(F.col("master_player_id").isNull()).count()
total_mappings = id_mapping.count()

print(f"\n✓ Total ID mappings: {total_mappings}")
print(f"⚠️  Unmapped source IDs: {unmapped_count}")

if unmapped_count > 0:
    print("\nSample unmapped players (likely data quality issues):")
    display(id_mapping.filter(F.col("master_player_id").isNull()).limit(10))

# Filter to only successfully mapped IDs
id_mapping_clean = id_mapping.filter(F.col("master_player_id").isNotNull())

# Write mapping table
id_mapping_clean.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.gold_player_id_mapping"
)

print(f"\n✓ Created table: {CATALOG}.{SCHEMA}.gold_player_id_mapping")

# Show sample mappings
print("\nSample ID mappings:")
display(spark.table(f"{CATALOG}.{SCHEMA}.gold_player_id_mapping").limit(20))

In [0]:
# Create consolidated gold weekly stats table with master player IDs
print("="*70)
print("CREATING GOLD WEEKLY STATS TABLE")
print("="*70)

# Load mapping table
id_mapping = spark.table(f"{CATALOG}.{SCHEMA}.gold_player_id_mapping")

# Load silver data with extracted metadata
silver_enriched = spark.sql("""
    SELECT 
        player_id as source_player_id,
        source,
        season,
        week,
        fantasy_points,
        stats,
        player_name,
        position,
        team,
        ingested_at
    FROM silver_enriched
""")

# Create aliases to avoid ambiguous column references
silver_alias = silver_enriched.alias("silver")
id_mapping_alias = id_mapping.select("master_player_id", "source", "source_player_id").alias("mapping")

# Join silver data to mapping table to get master player IDs
gold_stats = silver_alias.join(
    id_mapping_alias,
    (F.col("silver.source_player_id") == F.col("mapping.source_player_id")) &
    (F.col("silver.source") == F.col("mapping.source")),
    "left"
).select(
    F.col("mapping.master_player_id"),
    F.col("silver.source_player_id"),
    F.col("silver.source"),
    F.col("silver.season"),
    F.col("silver.week"),
    F.col("silver.fantasy_points"),
    F.col("silver.stats"),
    F.col("silver.player_name"),
    F.col("silver.position"),
    F.col("silver.team"),
    F.col("silver.ingested_at")
).filter(
    F.col("master_player_id").isNotNull()
)

record_count = gold_stats.count()
print(f"\n✓ Gold records: {record_count}")

# Write gold stats table
gold_stats.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.gold_weekly_stats"
)

print(f"\n✓ Created table: {CATALOG}.{SCHEMA}.gold_weekly_stats")

# Show summary
print("\nGold table summary:")
summary = spark.sql(f"""
    SELECT 
        source,
        COUNT(*) as records,
        COUNT(DISTINCT master_player_id) as unique_players,
        COUNT(DISTINCT season) as seasons,
        COUNT(DISTINCT week) as weeks,
        MIN(week) as min_week,
        MAX(week) as max_week
    FROM {CATALOG}.{SCHEMA}.gold_weekly_stats
    GROUP BY source
    ORDER BY records DESC
""")

display(summary)

In [0]:
%sql
-- Data Quality Checks for Gold Layer

-- 1. Check for players appearing in multiple sources
SELECT 
    dim.display_name,
    dim.position,
    dim.current_team,
    dim.source_count,
    dim.sources
FROM main.fantasai.gold_player_dim dim
WHERE dim.source_count >= 3
ORDER BY dim.source_count DESC, dim.display_name
LIMIT 50;

-- 2. Weekly stats completeness by source
SELECT 
    season,
    week,
    source,
    COUNT(DISTINCT master_player_id) as unique_players,
    COUNT(*) as total_records
FROM main.fantasai.gold_weekly_stats
GROUP BY season, week, source
ORDER BY season DESC, week DESC, source;

-- 3. Players with most complete data across sources
SELECT 
    dim.display_name,
    dim.position,
    dim.current_team,
    COUNT(DISTINCT stats.source) as sources_count,
    COLLECT_SET(stats.source) as sources,
    COUNT(*) as total_weekly_records
FROM main.fantasai.gold_player_dim dim
JOIN main.fantasai.gold_weekly_stats stats 
    ON dim.master_player_id = stats.master_player_id
GROUP BY dim.display_name, dim.position, dim.current_team
HAVING sources_count >= 3
ORDER BY sources_count DESC, total_weekly_records DESC
LIMIT 50;